# 03 — Portfolio Construction

Compare **HRP** vs **mean-variance** vs **minimum-variance** vs **risk-parity** vs **equal-weight** on an S&P-100-style basket, then backtest each with mandatory transaction costs and rank by Sharpe.

In [1]:
import sys, os, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Put the quantcortex repo root on the path (notebooks live in research/).
for p in ("..", "."):
    ap = os.path.abspath(p)
    if ap not in sys.path:
        sys.path.insert(0, ap)

RNG = np.random.default_rng(42)

def load_prices(symbols, start="2018-01-01", periods=1800):
    """Real prices via yfinance if available, else deterministic synthetic GBM."""
    try:
        from data.providers.yfinance_provider import YFinanceProvider
        px = YFinanceProvider().get_prices(list(symbols), start=start)
        if px is not None and not px.empty and px.shape[0] > 120:
            return px.dropna(how="all").ffill().dropna()
        raise RuntimeError("empty frame")
    except Exception as exc:
        print(f"[offline] yfinance unavailable ({type(exc).__name__}); using synthetic data.")
    dates = pd.bdate_range(start, periods=periods)
    mu = RNG.normal(0.0003, 0.00015, len(symbols))
    vol = RNG.uniform(0.008, 0.018, len(symbols))
    shocks = RNG.normal(mu, vol, size=(periods, len(symbols)))
    return pd.DataFrame(100 * np.exp(np.cumsum(shocks, axis=0)),
                        index=dates, columns=list(symbols))

def synth_ohlcv(close):
    """Build an OHLCV frame around a close series (synthetic intrabar range)."""
    close = close.dropna()
    hi = close * (1 + np.abs(RNG.normal(0, 0.004, len(close))))
    lo = close * (1 - np.abs(RNG.normal(0, 0.004, len(close))))
    op = close.shift(1).fillna(close.iloc[0])
    vol = RNG.integers(1_000_000, 6_000_000, len(close)).astype(float)
    return pd.DataFrame({"open": op, "high": hi, "low": lo, "close": close, "volume": vol})

print("quantcortex research environment ready.")


quantcortex research environment ready.


In [2]:
symbols = ["AAPL","MSFT","AMZN","NVDA","GOOGL","JPM","XOM","JNJ","PG","KO","UNH","HD"]
prices = load_prices(symbols)
returns = prices.pct_change().dropna()
print("basket:", len(symbols), "names |", prices.shape[0], "days")

[offline] yfinance unavailable (ImportError); using synthetic data.
basket: 12 names | 1800 days


## Fit the optimizers

In [3]:
from portfolio.equal_weight import EqualWeight
from portfolio.mean_variance import MeanVariance
from portfolio.minimum_variance import MinimumVariance
from portfolio.risk_parity import RiskParity
from portfolio.hrp import HierarchicalRiskParity

optimizers = {
    "EqualWeight": EqualWeight(),
    "MeanVariance": MeanVariance(),
    "MinVariance": MinimumVariance(),
    "RiskParity": RiskParity(),
    "HRP": HierarchicalRiskParity(),
}
weights = {name: opt.optimize(returns) for name, opt in optimizers.items()}
wdf = pd.DataFrame(weights, index=symbols)
for name, w in weights.items():
    assert abs(w.sum() - 1.0) < 1e-6 and (w >= -1e-9).all()
print("all optimizers honor the weight contract (sum=1, long-only)")
wdf.round(3)

all optimizers honor the weight contract (sum=1, long-only)


,EqualWeight,MeanVariance,MinVariance,RiskParity,HRP
AAPL,0.083,0.000,0.065,0.075,0.066
MSFT,0.083,0.233,0.056,0.069,0.054
AMZN,0.083,0.000,0.083,0.085,0.088
NVDA,0.083,0.537,0.127,0.107,0.129
GOOGL,0.083,0.000,0.085,0.086,0.079
JPM,0.083,0.000,0.178,0.128,0.201
XOM,0.083,0.000,0.059,0.070,0.054
JNJ,0.083,0.000,0.075,0.079,0.070
PG,0.083,0.000,0.066,0.074,0.064
KO,0.083,0.000,0.105,0.096,0.104


## Backtest each allocation (costs mandatory)

In [4]:
from backtest.costs.transaction_costs import TransactionCostModel
from backtest.engines.vectorized import VectorizedBacktest

rebal = prices.resample("MS").first().index            # monthly rebalance
rebal = rebal[(rebal >= prices.index[0]) & (rebal <= prices.index[-1])]
cost_model = TransactionCostModel()
results, equity = {}, {}
for name, w in weights.items():
    panel = pd.DataFrame([w] * len(rebal), index=rebal, columns=symbols)
    res = VectorizedBacktest(cost_model).run(panel, prices)
    results[name] = res.summary(); equity[name] = res.equity_curve
summary = pd.DataFrame(results).T[["cagr","ann_vol","sharpe","max_drawdown"]]
summary.sort_values("sharpe", ascending=False).round(3)

,cagr,ann_vol,sharpe,max_drawdown
HRP,0.112,0.061,1.777,-0.065
MinVariance,0.112,0.061,1.775,-0.070
MeanVariance,0.236,0.126,1.747,-0.105
RiskParity,0.111,0.062,1.720,-0.083
EqualWeight,0.109,0.066,1.600,-0.100


In [5]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
pd.DataFrame(equity).plot(ax=ax[0], lw=1.2); ax[0].set_title("Equity curves"); ax[0].set_ylabel("NAV")
wdf[["HRP","MeanVariance"]].plot(kind="bar", ax=ax[1]); ax[1].set_title("HRP vs MV weights")
plt.tight_layout(); print("portfolio construction complete.")

portfolio construction complete.
